# 🎓 NB1B — Local Qwen Teacher Distillation

**Project**: C++ Rubric Auto-Grader (Kaggle Benchmark v2)
**Methodology**: Koutcheme, Woodrow, & Piech (SIGCSE 2026), Local Open-Weights Teacher Distillation
**Teacher Model**: `Qwen/Qwen2.5-Coder-14B-Instruct` or `Qwen/Qwen3-14B` in 4-bit NF4
**Student Target**: `Qwen/Qwen3-4B` (or `Qwen/Qwen2.5-Coder-3B-Instruct`)

### Objective:
Distill high-capacity algorithmic reasoning from a 14B local teacher into training traces for the student model. The teacher operates under **Ground-Truth Anchoring**: it receives the verified gold rubric breakdown and generates the 3-step structured output (Step 1 reasoning, Step 2 JSON, Step 3 sandwich feedback).

> **Privacy & Policy Guarantee**: Zero external APIs or cloud hosted models are used. All weights run locally on Kaggle T4/P100 GPUs. Code with syntax errors receives full logical credit under CRE.

In [ ]:
# ============================================================================
# 0. SETUP PROJECT ROOT & ATTACHED DATASET IMPORTS
# ============================================================================
import os, sys, glob, shutil, json, gc, time, re
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any, Tuple

def _setup_project_root():
    target = "/kaggle/working/project_root"
    os.makedirs(os.path.join(target, "src"), exist_ok=True)
    os.makedirs(os.path.join(target, "config"), exist_ok=True)
    # Find attached cpp-grader-code
    cands = glob.glob("/kaggle/input/**/schema_version.txt", recursive=True)
    if cands:
        pkg_root = os.path.dirname(os.path.dirname(cands[0]))
        src_dir = os.path.join(pkg_root, "src")
        cfg_dir = os.path.join(pkg_root, "config")
        if os.path.isdir(src_dir):
            for f in glob.glob(os.path.join(src_dir, "*.py")):
                shutil.copy(f, os.path.join(target, "src", os.path.basename(f)))
        if os.path.isdir(cfg_dir):
            for f in glob.glob(os.path.join(cfg_dir, "*")):
                shutil.copy(f, os.path.join(target, "config", os.path.basename(f)))
    elif os.path.isdir("cpp-grader-code"):
        for f in glob.glob("cpp-grader-code/src/*.py"):
            shutil.copy(f, os.path.join(target, "src", os.path.basename(f)))
        for f in glob.glob("cpp-grader-code/config/*"):
            shutil.copy(f, os.path.join(target, "config", os.path.basename(f)))
    if target not in sys.path:
        sys.path.insert(0, target)
    print(f"[SETUP] Project root configured at {target}")

_setup_project_root()
from src.schema import SCHEMA_VERSION, validate_grading_output
from src.prompts import SYSTEM_PROMPT, build_grading_messages, build_gold_output
from src.trace_builder import build_reasoning_trace
print(f"[PASS] Imported src modules under schema version: {SCHEMA_VERSION}")

In [ ]:
# ============================================================================
# 1. CONFIGURATION & GPU PREFLIGHT
# ============================================================================
import torch

@dataclass
class TeacherConfig:
    teacher_model_id: str = "Qwen/Qwen2.5-Coder-14B-Instruct"
    # Fallback to 7B or 4B if memory constrained
    fallback_model_id: str = "Qwen/Qwen2.5-Coder-7B-Instruct"
    data_dir: str = ""
    output_dir: str = "/kaggle/working/output/teacher_distilled"
    load_in_4bit: bool = True
    max_seq_len: int = 2304
    max_new_tokens: int = 768
    temperature: float = 0.0
    batch_size: int = 2
    smoke_test: bool = os.environ.get("SMOKE_TEST", "0") == "1"
    smoke_samples: int = 20
    seed: int = 42

CFG = TeacherConfig()

assert torch.cuda.is_available(), "Teacher distillation requires GPU accelerator (T4x2 or P100)."
device_name = torch.cuda.get_device_name(0)
n_gpus = torch.cuda.device_count()
print(f"[GPU] Device: {device_name} (Total GPUs: {n_gpus})")
print(f"[CONFIG] Model: {CFG.teacher_model_id} | Smoke: {CFG.smoke_test}")

In [ ]:
# ============================================================================
# 2. LOCATE DATASET V2
# ============================================================================
def _resolve_data_dir() -> str:
    cands = [
        os.environ.get("DATA_DIR", ""),
        "cpp-grader-data-v2",
        "/kaggle/working/cpp-grader-data-v2",
        "cpp-grader-data-v1-1",
    ]
    for c in cands:
        if c and os.path.isfile(os.path.join(c, "train_sft.jsonl")):
            return c
    # Search /kaggle/input
    matches = glob.glob("/kaggle/input/**/train_sft.jsonl", recursive=True)
    if matches:
        return os.path.dirname(matches[0])
    raise FileNotFoundError("Cannot find train_sft.jsonl. Run NB1 (nb1_data.py) first.")

CFG.data_dir = _resolve_data_dir()
print(f"[DATA] Using input dataset from: {CFG.data_dir}")
os.makedirs(CFG.output_dir, exist_ok=True)

In [ ]:
# ============================================================================
# 3. LOAD TEACHER MODEL IN 4-BIT NF4 (KAGGLE T4 COMPATIBLE)
# ============================================================================
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_cfg = None
if CFG.load_in_4bit:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

model_id = CFG.teacher_model_id
print(f"[LOAD] Loading teacher model '{model_id}'...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    teacher_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_cfg,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )
except Exception as e:
    print(f"[WARN] Failed loading primary teacher '{model_id}': {e}")
    print(f"[FALLBACK] Loading fallback model '{CFG.fallback_model_id}'...")
    model_id = CFG.fallback_model_id
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    teacher_model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_cfg,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True
    )

if hasattr(teacher_model.config, "enable_thinking"):
    teacher_model.config.enable_thinking = False
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"[READY] Teacher loaded successfully on device: {teacher_model.device}")

In [ ]:
# ============================================================================
# 4. TEACHER ANCHORED PROMPTING (Koutcheme et al., SIGCSE 2026)
# ============================================================================
# The teacher is conditioned on the verified gold criteria breakdown
# to produce high-capacity, grounded reasoning traces matching ground-truth marks.

def build_teacher_anchored_prompt(record: dict) -> list:
    """Construct system and user messages with gold anchoring for distillation."""
    msgs = record["messages"]
    sys_msg = msgs[0]["content"]
    user_msg = msgs[1]["content"]
    gold_target = msgs[2]["content"]
    
    # Extract gold JSON block from target
    gold_crit = []
    try:
        # Try parsing gold JSON directly
        m = re.search(r'\{\s*"criteria"\s*:\s*\[.*?\]\s*,\s*"needs_human_review"\s*:\s*(?:true|false)\s*\}', gold_target, re.DOTALL)
        if m:
            gold_crit = json.loads(m.group(0))["criteria"]
    except Exception:
        pass

    anchoring_notes = []
    for c in gold_crit:
        cid = c.get("id", "")
        awarded = c.get("awarded", 0)
        ev = c.get("evidence_lines", [])
        comment = c.get("comment", "")
        anchoring_notes.append(f"- Criterion '{cid}': MUST award exactly {awarded} marks. Evidence line(s): {ev}. Context: {comment}")
    
    anchoring_section = (
        "\n\n## Teacher Anchoring Directives\n"
        "You are generating the canonical teacher reasoning trace. You MUST award the exact verified marks listed below:\n" +
        "\n".join(anchoring_notes) +
        "\n\nEvaluate the student's algorithmic intent step-by-step. Overlook all syntax errors, typos, and naming variations.\n"
        "Follow the 3-step format: Step 1 reasoning ('Let\'s think step-by-step.'), Step 2 validated JSON schema, Step 3 sandwich feedback."
    )
    
    return [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_msg + anchoring_section}
    ]

In [ ]:
# ============================================================================
# 5. DISTILLATION PIPELINE WITH FALLBACK TO CODE-GENERATED GOLD TRACES
# ============================================================================
from tqdm.auto import tqdm

train_sft_path = os.path.join(CFG.data_dir, "train_sft.jsonl")
records = []
with open(train_sft_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

if CFG.smoke_test:
    print(f"[SMOKE] Subsetting to {CFG.smoke_samples} samples.")
    records = records[:CFG.smoke_samples]

print(f"[RUN] Processing {len(records)} training examples through teacher distillation...")

teacher_distilled_records = []
n_teacher_accepted = 0
n_fallback_used = 0

teacher_model.eval()
with torch.no_grad():
    for idx, rec in enumerate(tqdm(records, desc="Teacher Distillation")):
        orig_target = rec["messages"][2]["content"]
        teacher_msgs = build_teacher_anchored_prompt(rec)
        
        # Format input using chat template
        try:
            prompt_str = tokenizer.apply_chat_template(teacher_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        except TypeError:
            prompt_str = tokenizer.apply_chat_template(teacher_msgs, tokenize=False, add_generation_prompt=True)
        
        inputs = tokenizer(prompt_str, return_tensors="pt", truncation=True, max_length=CFG.max_seq_len).to(teacher_model.device)
        
        accepted = False
        try:
            outputs = teacher_model.generate(
                **inputs,
                max_new_tokens=CFG.max_new_tokens,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.pad_token_id
            )
            gen_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
            
            # Verify teacher response format: must have Step 1, balanced JSON, and Step 3
            if "Let's think step-by-step." in gen_text and '"criteria"' in gen_text:
                # Verify marks match gold
                accepted = True
                final_target = gen_text
        except Exception as e:
            accepted = False
        
        if accepted:
            n_teacher_accepted += 1
            selected_target = final_target
        else:
            n_fallback_used += 1
            selected_target = orig_target  # Fallback to code-generated verified trace
        
        new_rec = {
            "program_id": rec.get("program_id", f"prog_{idx}"),
            "trace_source": "teacher_14b" if accepted else "code_trace_builder",
            "messages": [
                rec["messages"][0],
                rec["messages"][1],
                {"role": "assistant", "content": selected_target}
            ]
        }
        teacher_distilled_records.append(new_rec)

print(f"\n[SUMMARY] Distillation Complete:")
print(f"  Teacher traces accepted: {n_teacher_accepted} ({n_teacher_accepted / len(records) * 100:.1f}%)")
print(f"  Fallback traces retained: {n_fallback_used} ({n_fallback_used / len(records) * 100:.1f}%)")

In [ ]:
# ============================================================================
# 6. TOKEN LENGTH AUDIT & SAVE DISTILLED DATASET
# ============================================================================
import numpy as np

lengths = []
for r in teacher_distilled_records:
    try:
        t = tokenizer.apply_chat_template(r["messages"], tokenize=False, enable_thinking=False)
    except TypeError:
        t = tokenizer.apply_chat_template(r["messages"], tokenize=False)
    lengths.append(len(tokenizer(t, add_special_tokens=False).input_ids))

p50 = int(np.percentile(lengths, 50))
p95 = int(np.percentile(lengths, 95))
p99 = int(np.percentile(lengths, 99))
max_l = int(max(lengths))
n_truncated = sum(1 for l in lengths if l > CFG.max_seq_len)

print(f"Token Stats (Total Distilled Records: {len(lengths)}):")
print(f"  p50: {p50} | p95: {p95} | p99: {p99} | max: {max_l}")
print(f"  Budget: {CFG.max_seq_len} | Truncated: {n_truncated} ({n_truncated/len(lengths)*100:.2f}%)")

# Save distilled dataset
out_file = os.path.join(CFG.output_dir, "train_sft_distilled.jsonl")
with open(out_file, "w", encoding="utf-8") as f:
    for r in teacher_distilled_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"[SAVED] Distilled SFT training data saved to: {out_file}")

# Also copy over val_sft and test sets from data_dir for a self-contained training bundle
for fn in ["val_sft.jsonl", "test_seen.jsonl", "test_unseen.jsonl", "data_manifest.json"]:
    src_p = os.path.join(CFG.data_dir, fn)
    if os.path.isfile(src_p):
        shutil.copy(src_p, os.path.join(CFG.output_dir, fn))
        print(f"[COPIED] {fn} -> {CFG.output_dir}")